# AES guru 300 · RoBERTa, BGE-M3, fusion → prediksi dan evaluasi

Urutan mengikuti **BGE-M3 [AES].ipynb** unggahan: instalasi/impor → Excel/dataset → split/tokenisasi → embedding → arsitektur → loss/optimizer → training → prediksi/evaluasi. **Kode proses ada di cell dengan komentar `#`; output tidak lagi mencetak source kode.**

**S1** RoBERTa Indonesia fine-tuned. **S2** BGE-M3 frozen+MLP. **S3** fusion RoBERTa fine-tuned dan BGE frozen. Ini mempertahankan rancangan sebelumnya; notebook BGE unggahan sendiri memperbarui seluruh BGE. Dataset hanya mempunyai satu `rater`, jadi setiap model menghasilkan **satu skor**, bukan empat label rubrik yang tidak ada.

Gunakan GPU Colab, Python 3.12/3.13. Jalankan berurutan. Default 5 epoch/fold,5 fold,15 CVfit+3 refit; proses bisa melewati sesi gratis. **Hasil akhir dirangkum dalam lima Excel** dengan nama sheet dan kolom yang mudah dibaca. Rincian setiap tahap tetap tersimpan secara privat. Preview notebook hanya cuplikan. Seluruh jawaban: embedding per jawaban; satu jawaban contoh: vektor token/kata/kalimat lengkap. Definisi fungsi menyiapkan proses, bukan menghasilkan skor dengan sendirinya.

File dataset, Excel, checkpoint dan notebook terisi tetap privat. GitHub menyimpan source/notebook tanpa output.


## 1 · Ambil source terbaru
Source publik diambil; dataset tidak ditanam di GitHub.


In [ ]:
import os, sys, json, subprocess, html, secrets, atexit
from pathlib import Path
from IPython.display import display, HTML
REPO_URL = 'https://github.com/MalikHariyanto/Automated_Essay_Scoring.git'
BRANCH = 'codex/aes-learning'
IN_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()
ROOT = Path('/content/Automated_Essay_Scoring') if IN_COLAB else Path.cwd()
if not IN_COLAB:
    while not (ROOT/'src/aes').is_dir() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if not (ROOT/'.git').exists():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError('Folder target berisi file lain; kode tidak ditimpa.')
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(ROOT)],check=True)
elif IN_COLAB:
    branch = subprocess.check_output(['git','branch','--show-current'],cwd=ROOT,text=True).strip()
    dirty = subprocess.check_output(['git','status','--porcelain'],cwd=ROOT,text=True).strip()
    if branch != BRANCH or dirty:
        raise RuntimeError('Checkout mempunyai branch/perubahan lain; simpan dahulu atau pakai runtime baru.')
    subprocess.run(['git','pull','--ff-only','origin',BRANCH],cwd=ROOT,check=True)
print('Versi kode:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()[:12])
print('Versi Python:',sys.version.split()[0])


## 2 · Instalasi dan pemilih Excel

Setup memeriksa versi pustaka yang sudah tersedia. Versi yang cocok dipakai ulang; versi yang berbeda dipasang dalam lingkungan khusus AES tanpa mencoba menghapus pustaka bawaan. PyTorch runtime dipertahankan agar akses GPU yang tersedia tidak diganti wheel CPU. Pemeriksaan mencakup versi, dependensi pustaka yang dipakai AES, dan import; bukan klaim semua aplikasi bawaan Colab memakai versi yang sama.

Output utama berisi status singkat. **Rincian instalasi** bisa dibuka jika diperlukan. Jika pemeriksaan gagal, cell berhenti dengan alasan dan log, bukan menampilkan status siap. Python worker numerik terpisah menghindari NumPy lama yang sudah dimuat kernel.

Awalan **`%%aes` menjalankan kode cell pada worker yang sama**, sehingga variabel bertahan antar-cell. Upload Excel sekali per runtime; bila file sudah ada tidak diminta lagi.


In [ ]:
PYTHON = os.environ.get('AES300_PYTHON','')
setup_command=[sys.executable,str(ROOT/'scripts/setup_dataset300.py')]
if PYTHON:setup_command+=['--verify-python',PYTHON]
process = subprocess.Popen(setup_command,cwd=ROOT,
                           stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
log=[]
for line in process.stdout:
    print(line,end='',flush=True);log.append(line)
def show_installation_log():
    path=ROOT/'outputs/dataset300_setup.log'
    contents=path.read_text() if path.is_file() else ''.join(log)
    display(HTML('<details style="background:#f8fafc;color:#172035;padding:12px;border:1px solid #94a3b8">'
                 '<summary>Rincian instalasi — buka jika diperlukan</summary>'
                 '<pre style="max-height:320px;overflow:auto;white-space:pre-wrap;color:#172035">'
                 +html.escape(contents)+'</pre></details>'))
if process.wait():
    show_installation_log()
    raise RuntimeError('Persiapan AES belum berhasil. '+''.join(log[-4:]))
ready=json.loads((ROOT/'outputs/dataset300_setup.json').read_text())
if ready['runtime_version'] != list(sys.version_info[:2]) or not ready['imports_verified']:
    raise RuntimeError('Metadata setup tidak cocok runtime aktif.')
PYTHON=ready['python']
setup_rows=[
    ('Status','Siap — versi, dependensi, dan import AES lulus pemeriksaan'),
    ('Python',ready['python_version']),
    ('PyTorch',ready['torch']),
    ('Perangkat','GPU: '+ready['device'] if ready['cuda_available'] else 'CPU — GPU belum terdeteksi'),
    ('Pustaka AES','Versi yang cocok dipakai ulang' if ready['packages_action']=='reused'
                    else 'Versi khusus AES dipasang; pustaka bawaan tetap disimpan'),
    ('Pustaka utama','NumPy '+ready['versions']['numpy']+'; Transformers '+ready['versions']['transformers']),
    ('Model','Belum dimuat; tahap ini hanya menyiapkan lingkungan')]
status_html=''.join('<tr><th style="padding:9px;text-align:left;border:1px solid #94a3b8">'+html.escape(label)
                    +'</th><td style="padding:9px;border:1px solid #94a3b8">'+html.escape(value)+'</td></tr>'
                    for label,value in setup_rows)
display(HTML('<table style="background:#f8fafc!important;color:#172035!important;border-collapse:collapse">'
             +status_html+'</table>'))
show_installation_log()
DATASET_NAME='fix_dataset_300.xlsx'
PRIVATE_INPUT=ROOT/'data/dataset300';PRIVATE_INPUT.mkdir(parents=True,exist_ok=True)
DATA_PATH=Path(os.environ.get('AES300_DATASET',str(PRIVATE_INPUT/DATASET_NAME))).resolve()
if not DATA_PATH.exists():
    if not IN_COLAB:
        raise FileNotFoundError(str(DATA_PATH))
    from google.colab import files
    previous=Path.cwd()
    try:
        os.chdir(PRIVATE_INPUT);uploaded=files.upload()
    finally:
        os.chdir(previous)
    if DATASET_NAME not in uploaded:
        raise ValueError('Upload '+DATASET_NAME)
def display_aes_table(event):
    display(HTML(event['html']))
if '_aes_process' in globals() and _aes_process.poll() is None:
    _aes_process.terminate()
_aes_prefix='__AES_'+secrets.token_hex(12)+'__'
child_env={**os.environ,'PYTHONPATH':str(ROOT/'src'),'HF_HOME':str(ROOT/'outputs/cache/huggingface'),
           'AES300_DATASET':str(DATA_PATH),'TOKENIZERS_PARALLELISM':'false',
           'OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2','PYTHONUNBUFFERED':'1'}
_aes_process=subprocess.Popen([PYTHON,str(ROOT/'scripts/aes_lesson_worker.py'),'--prefix',_aes_prefix],
                              cwd=ROOT,env=child_env,stdin=subprocess.PIPE,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1)
while True:
    line=_aes_process.stdout.readline()
    if not line:
        raise RuntimeError('Worker gagal mulai; lihat log setup.')
    if line.startswith(_aes_prefix):
        print('Proses perhitungan siap. Python',json.loads(line[len(_aes_prefix):])['python']);break
    print(line,end='')
def aes_cell(line,cell):
    if _aes_process.poll() is not None:
        raise RuntimeError('Worker berhenti; jalankan setup dan cell berikutnya dari awal.')
    _aes_process.stdin.write(json.dumps({'code':cell,'filename':'<notebook-cell>'})+'\n')
    _aes_process.stdin.flush()
    while True:
        output=_aes_process.stdout.readline()
        if not output:
            raise RuntimeError('Worker terhenti; jika RAM penuh pilih runtime lebih besar.')
        if not output.startswith(_aes_prefix):
            print(output,end='',flush=True);continue
        event=json.loads(output[len(_aes_prefix):])
        if event['kind']=='done':break
        if event['kind']=='error':raise RuntimeError(event['traceback'])
        if event['kind']=='table':display_aes_table(event)
get_ipython().register_magic_function(aes_cell,'cell','aes')
atexit.register(lambda:_aes_process.terminate() if _aes_process.poll() is None else None)
print('Dataset siap. Jalankan cell %%aes berurutan.')


## 3 · Impor dan konfigurasi sebelum eksperimen

Sama seperti file unggahan: impor pustaka, pilih device, tentukan epoch/batch. Default 5 epoch/fold mengikuti rancangan sebelumnya, bukan 20 epoch file unggahan. **Output:**`05_Riwayat_Pelatihan.xlsx` dan `05_Riwayat_Pelatihan.xlsx`. Rujukan dipilih dari koleksi pengguna; tahun/versi dan batas penerapan dicatat, bukan klaim semua paper memakai model identik.


In [ ]:
%%aes
import os, sys, json, math, hashlib, random, warnings, re, itertools, unicodedata, zipfile, subprocess, gc, logging
from collections import Counter
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.functional import normalize
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from openpyxl import load_workbook
from aes.cleaning import clean
from aes.teacher_score import SCOPE
from aes.teacher_score.data import DEFAULTS, REQUIRED, Bundle, config, digest, dump, read_dataset, run_folder
from aes.teacher_score.excel import ExcelJournal, vector_frame
SCENARIOS=('S1','S2','S3')

# Nama checkpoint adalah alamat model pretrained di Hugging Face, bukan objek model yang sudah dimuat.
ROBERTA_CHECKPOINT='flax-community/indonesian-roberta-base'  # RoBERTa bahasa Indonesia untuk S1 dan S3.
BGE_CHECKPOINT='BAAI/bge-m3'  # BGE-M3 untuk menghasilkan embedding di S2 dan S3.
# Revision mengunci versi bobot/tokenizer agar eksperimen berikutnya memakai sumber yang sama.
ROBERTA_REVISION=DEFAULTS['roberta']['revision']
BGE_REVISION=DEFAULTS['bge']['revision']

# Ubah parameter DI SINI sebelum latihan; jangan memilihnya dari hasil data uji.
# Epoch = satu putaran data latih. Lima fold = lima pembagian latihan/validasi di data development.
EPOCHS,FOLDS=5,5
# Proses satu jawaban per microbatch, lalu akumulasikan gradien hingga batch efektif delapan jawaban.
MICRO_BATCH,EFFECTIVE_BATCH=1,8
NEW_ANSWER=''  # Kosong: satu jawaban held-out untuk contoh scoring/vektor.
AUDIT_ONLY=os.environ.get('AES300_AUDIT_ONLY')=='1'
RUN_MODELS=RUN_TRAINING=RUN_WORD_VECTORS=RUN_FINAL_FEATURES=not AUDIT_ONLY
cfg=config(dict(dataset=os.environ['AES300_DATASET'],output=str(ROOT/'outputs/dataset300_inline'),
                sheet='Sheet1',expected_rows=300,score_min=0,score_max=100,
                epochs=EPOCHS,folds=FOLDS,micro_batch=MICRO_BATCH,effective_batch=EFFECTIVE_BATCH,
                roberta={'checkpoint':ROBERTA_CHECKPOINT,'revision':ROBERTA_REVISION},
                bge={'checkpoint':BGE_CHECKPOINT,'revision':BGE_REVISION},
                device='auto',new_answer=NEW_ANSWER,vector_source='pretrained',lesson_version='inline_excel_v1',
                runtime_versions={p:version(p) for p in ('torch','transformers','numpy')}))
bundle=read_dataset(cfg)  # Periksa enam header, label, duplikat dan kelompok email.
output=run_folder(cfg,bundle)
journal=ExcelJournal(output/'excel',{'scope':SCOPE,'dataset_sha256':bundle.source_sha256,
                     'run_folder':str(output),'source_commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip(),
                     'privacy':'PRIVAT; bukan untuk repo publik'})
journal.save('01_konfigurasi',{'Parameter':pd.DataFrame([{'name':k,'value':v} for k,v in cfg.items()]),
                             'Versi':pd.DataFrame([{'library':p,'version':version(p)} for p in ('torch','transformers','numpy','pandas','scikit-learn','openpyxl')])},
             'Konfigurasi aktual; belum ada hasil training')
references=json.loads((ROOT/'docs/aes_dataset300_references.json').read_text())
journal.save('01_rujukan_paper',{'Rujukan':pd.DataFrame(references)},'Rujukan koleksi pengguna dan batas penerapan')
show(pd.DataFrame(references)[['id','title','used_for']],'Rujukan proses')
print('Perangkat perhitungan:', 'GPU (CUDA)' if torch.cuda.is_available() else 'CPU')


## 4 · Baca Excel dan cleaning

Hanya `response_text` masuk encoder. Soal/kunci/email/skor tidak menjadi fitur. Cleaning mempertahankan ejaan, kapital, tanda baca/paragraf; sumber tidak ditimpa. Dataset satu nilai tidak memberi empat label rubrik; file bertopik metodologi penelitian ini juga tidak membuktikan populasi SMAkelas XI.

**Output:** dataset asli, audit, teks sebelum/sesudah dan target loss `(rater−0)/100`→`01_Data_dan_Tokenisasi.xlsx`. Rujukan m01 menempatkan pemahaman/persiapan data sebelum modeling.


In [ ]:
%%aes
df=pd.read_excel(cfg['dataset'],sheet_name=cfg['sheet'])
df=df.loc[:,~df.columns.astype(str).str.startswith('Unnamed:')]
cleaning=pd.DataFrame([{'row_id':r['row_id'],'teks_asli':r['raw_text'],'teks_model':r['text'],
                       'berubah':r['raw_text']!=r['text'],'skor_guru':r['score'],
                       'target_loss':(r['score']-cfg['score_min'])/(cfg['score_max']-cfg['score_min'])} for r in bundle.rows])
journal.save('02_dataset_dan_cleaning',{'Dataset_asli':df,'Audit':pd.DataFrame([bundle.summary]),
                                      'Sebelum_sesudah':cleaning},'Pembacaan/cleaning aktual; sumber utuh')
show(cleaning[['row_id','teks_model','skor_guru']],'Contoh jawaban dan nilai guru')
print(len(df),'jawaban tersedia. Setiap jawaban memiliki satu nilai guru; setiap model menghasilkan satu prediksi skor.')


## 5 · Split 80/20 dan 5-fold validasi

Email/duplikat tidak menyeberang split. Kandidat dipilih mendekati 80/20 berdasarkan jumlah baris saja, tanpa melihat skor/model. Lima fold berada di development; tidak diperlukan split validasi ketiga. **Output:**`01_Data_dan_Tokenisasi.xlsx`, setiapbaris/kelompok/fold. Ini menguji jawaban kelompok baru pada pertanyaan yang sama, bukan unseen-prompt. Rujukan m09 membahas dependensi fold, bukan kewajiban universal 5 fold.


In [ ]:
%%aes
def make_split(bundle, cfg):
    indices = np.arange(len(bundle.rows))
    splitter = GroupShuffleSplit(n_splits=cfg['split_candidates'], test_size=cfg['test_fraction'], random_state=cfg['split_seed'])
    # Pilih kedekatan JUMLAH BARIS dengan target 80:20, bukan skor/hasil model.
    # GroupShuffleSplit membagi kelompok; ukuran kelompok tidak selalu sama.
    candidates = list(splitter.split(indices, groups=bundle.groups))
    development, test = min(candidates, key=lambda pair: abs(len(pair[1])/len(indices)-cfg['test_fraction']))
    if len(set(bundle.groups[development])) < cfg['folds']:
        raise ValueError('Kelompok training tidak cukup untuk CV yang diminta.')
    folds = []
    for number, (a, b) in enumerate(GroupKFold(n_splits=cfg['folds']).split(development, groups=bundle.groups[development]), 1):
        training, validation = development[a], development[b]
        assert not set(bundle.groups[training]) & set(bundle.groups[validation])
        folds.append({'fold': number, 'train': training.tolist(), 'validation': validation.tolist()})
    assert not set(bundle.groups[development]) & set(bundle.groups[test])
    manifest = {'scope': SCOPE, 'source_sha256': bundle.source_sha256,
                'development': development.tolist(), 'test': test.tolist(), 'folds': folds,
                'development_rows': len(development), 'test_rows': len(test),
                'development_groups': len(set(bundle.groups[development])),
                'test_groups': len(set(bundle.groups[test])),
                'target_test_row_fraction': cfg['test_fraction'],
                'candidate_group_fraction': cfg['test_fraction'],
                'split_candidates': cfg['split_candidates'],
                'selection_uses_labels': False,
                'actual_test_row_fraction': len(test)/len(indices),
                'policy': 'Holdout kelompok: pilih kandidat deterministik terdekat target proporsi baris berdasarkan ukuran kelompok saja, tanpa label/hasil model. CV hanya di development. Email/duplikat tidak menyeberang. Mengukur kelompok baru pada pertanyaan ini, bukan unseen-prompt.',
                'test_used_for_selection': False}
    manifest['split_sha256'] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    return manifest

manifest=make_split(bundle,cfg)
dump(output/'split_manifest.json',manifest)
roles={i:'development' for i in manifest['development']}
roles.update({i:'test' for i in manifest['test']})
split_rows=pd.DataFrame([{'row_id':r['row_id'],'role':roles[i],'connected_group':int(bundle.groups[i])} for i,r in enumerate(bundle.rows)])
fold_rows=pd.DataFrame([{'fold':f['fold'],'row_id':bundle.rows[i]['row_id'],'role':role} for f in manifest['folds'] for role,key in [('train','train'),('validation','validation')] for i in f[key]])
journal.save('03_split_dan_cv',{'Split_setiap_baris':split_rows,'Fold_setiap_baris':fold_rows,
                              'Ringkasan':pd.DataFrame([{k:v for k,v in manifest.items() if k not in ('folds','development','test')}])},'Split dan5fold sama untukS1/S2/S3')
show(split_rows.groupby('role').size().rename('rows').reset_index(),'Pembagian aktual')
trace_index=manifest['test'][cfg['trace_position']]
trace_text=cfg.get('new_answer','').strip() or bundle.rows[trace_index]['text']
from aes.teacher_score.presentation import friendly_value
print('Jawaban contoh:',friendly_value('row_id',bundle.rows[trace_index]['row_id']),'. Contoh ini dipakai untuk melihat token, vektor, dan prediksi setelah model dipilih.')


## 6 · Tokenizer dan window yang benar-benar dipakai

RoBERTa 512=510 isi+2 special, overlap 128, seluruh token akhir tercakup. BGE memakai tokenizer berbeda, batas eksperimen 1024 termasukspecial; truncation dilaporkan. Dasar tokenizer/encoder: m04/p14; batas/overlap ini keputusan proyek. **Output:**token,ID,attentionmask,padding,window/coverage→`01_Data_dan_Tokenisasi.xlsx`.


In [ ]:
%%aes
# Buat window, tambahkan special sekali, mean CLS secara differentiable.
# Membagi token isi yang panjang menjadi beberapa potongan; bagian yang bertumpang tindih menjaga konteks.
def content_windows(text, tokenizer, maximum=512, overlap=128):
    capacity = maximum - tokenizer.num_special_tokens_to_add(pair=False)
    if not 0 <= overlap < capacity: raise ValueError('Overlap harus lebih kecil dari content capacity')
    # Tokenizer mengubah teks menjadi ID subword, bukan menghitung skor.
    ids = tokenizer.encode(text, add_special_tokens=False)
    starts = [0]
    while starts[-1]+capacity < len(ids): starts.append(starts[-1]+capacity-overlap)
    chunks=[]
    for start in starts:
        content=ids[start:start+capacity]
        prepared=tokenizer.prepare_for_model(content, add_special_tokens=True, truncation=False,
                                             return_attention_mask=True, return_token_type_ids=False)
        assert len(prepared['input_ids']) <= maximum
        chunks.append(dict(prepared))
    return chunks, {'content_tokens':len(ids),'capacity':capacity,'stride':capacity-overlap,
                    'starts':starts,'windows':len(chunks)}


# Menambahkan token khusus dan padding, lalu mengembalikan tensor input beserta pemetaan ke jawaban asal.
def batch_windows(texts, tokenizer, device='cpu', maximum=512, overlap=128):
    chunks=[]; mapping=[]; stats=[]
    for i,text in enumerate(texts):
        c,s=content_windows(text,tokenizer,maximum,overlap); chunks.extend(c); mapping.extend([i]*len(c));stats.append(s)
    batch=tokenizer.pad(chunks, padding=True, return_tensors='pt')
    return {k:v.to(device) for k,v in batch.items()}, torch.tensor(mapping,device=device), stats


def mean_by_essay(cls, mapping, n):
    sums=cls.new_zeros((n,cls.shape[-1])).index_add(0,mapping,cls)
    counts=cls.new_zeros(n).index_add(0,mapping,cls.new_ones(len(mapping)))
    if (counts==0).any(): raise ValueError('Esai tanpa window')
    return sums / counts[:,None]


In [ ]:
%%aes
# Checkpoint terkunci; RoBERTa requires_grad=True, BGE frozen.
def device_name(cfg):
    requested = cfg['device']
    if requested == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA diminta tetapi tidak tersedia; pilih GPU di Colab atau device=cpu.')
    return ('cuda' if torch.cuda.is_available() else 'cpu') if requested == 'auto' else requested


def release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class ExpectedPoolerNotice(logging.Filter):
    """Hanya saring pemberitahuan dua bobot pooler yang memang tidak dipakai."""
    def filter(self, record):
        import ast
        import re
        message = record.getMessage()
        if not message.startswith('Some weights of the model checkpoint at ') or 'were not used when initializing' not in message:
            return True
        match = re.search(r"\['[^\n]+?\]", message)
        if match is None:
            return True
        try:
            keys = ast.literal_eval(match.group())
        except (ValueError, SyntaxError):
            return True
        return set(keys) != {'pooler.dense.bias', 'pooler.dense.weight'}


# Memuat encoder dari nama checkpoint dan revision pada cfg; hasilnya objek model pembaca teks.
def encoder(cfg, role):
    # Ambil nama checkpoint dan revision model yang dipilih: roberta atau bge.
    spec = cfg[role]
    logger = logging.getLogger('transformers.modeling_utils')
    notice = ExpectedPoolerNotice()
    logger.addFilter(notice)
    try:
        # Muat arsitektur dan bobot pretrained. Ini baru inisialisasi encoder, belum latihan memakai nilai guru.
        result = AutoModel.from_pretrained(spec['checkpoint'], revision=spec['revision'], add_pooling_layer=False)
    finally:
        logger.removeFilter(notice)
    # Periksa panjang vektor: RoBERTa 768 angka, BGE-M3 1024 angka.
    expected = 768 if role == 'roberta' else 1024
    if result.config.hidden_size != expected:
        raise ValueError(f'{role}: dimensi checkpoint tidak cocok dengan {expected}.')
    # Parameter RoBERTa ikut diperbarui; parameter BGE tidak ikut diperbarui.
    # RoBERTa boleh belajar; bobot encoder BGE tidak diperbarui.
    result.requires_grad_(role == 'roberta')
    return result


# Memuat pemecah teks milik checkpoint yang sama. Hasil tokenisasi menjadi input encoder.
def tokenizer(cfg, role, fast=False):
    # Muat tokenizer dari checkpoint yang sama agar ID token cocok dengan kosakata model.
    result = AutoTokenizer.from_pretrained(cfg[role]['checkpoint'], revision=cfg[role]['revision'],
                                           use_fast=fast if role == 'roberta' else True)
    if role == 'roberta' and not fast and result.num_special_tokens_to_add(False) != 2:
        raise ValueError('RoBERTa slow tokenizer harus menambahkan dua special token.')
    return result


In [ ]:
%%aes
def export_tokens(bundle, cfg, journal):
    rows, statistics = [], []
    roberta_tok, bge_tok = tokenizer(cfg, 'roberta'), tokenizer(cfg, 'bge')
    # Rekam input window RoBERTa beserta padding/attention mask aktual per jawaban.
    for index, text in enumerate(bundle.texts):
        windows, mapping, stats = batch_windows([text], roberta_tok, 'cpu', cfg['roberta_max_length'], cfg['roberta_overlap'])
        row_id = bundle.rows[index]['row_id']
        statistics.append({'row_id': row_id, 'model': 'roberta', **stats[0]})
        for window, ids in enumerate(windows['input_ids'].tolist()):
            masks = windows['attention_mask'][window].tolist()
            pieces = roberta_tok.convert_ids_to_tokens(ids)
            for position, (identifier, token, mask) in enumerate(zip(ids, pieces, masks)):
                rows.append({'row_id': row_id, 'window': window, 'position': position,
                             'token_id': identifier, 'token': token, 'attention_mask': mask})
    roberta_frame = pd.DataFrame(rows)
    rows = []
    # Urutan/batch/padding BGE sama dengan ekstraksi fitur frozen sesudah tahap ini.
    for start in range(0, len(bundle.rows), cfg['bge_batch']):
        texts = bundle.texts[start:start+cfg['bge_batch']]
        encoded = bge_tok(texts, padding=True, truncation=True, max_length=cfg['bge_max_length'], return_tensors='pt')
        for local, ids in enumerate(encoded['input_ids'].tolist()):
            row_id = bundle.rows[start+local]['row_id']
            full_length = len(bge_tok.encode(texts[local], add_special_tokens=True))
            statistics.append({'row_id': row_id, 'model': 'bge', 'tokens_before_truncation': full_length,
                               'kept_including_special': int(encoded['attention_mask'][local].sum()),
                               'truncated': full_length > cfg['bge_max_length']})
            for position, (identifier, token, mask) in enumerate(zip(ids, bge_tok.convert_ids_to_tokens(ids), encoded['attention_mask'][local].tolist())):
                rows.append({'row_id': row_id, 'position': position, 'token_id': identifier,
                             'token': token, 'attention_mask': mask})
    journal.save('04_tokenisasi', {'RoBERTa_input': roberta_frame, 'BGE_input': pd.DataFrame(rows),
                                 'Panjang_dan_coverage': pd.DataFrame(statistics)}, 'Token/ID/window/padding aktual',
                 source={'roberta': cfg['roberta'], 'bge': cfg['bge']})
    return roberta_frame, pd.DataFrame(rows)

if RUN_MODELS:
    token_roberta,token_bge=export_tokens(bundle,cfg,journal)
    show(token_roberta,'Token RoBERTa');show(token_bge,'Token BGE-M3')
else:
    print('Pemeriksaan data saja: tokenisasi dan model belum dijalankan.')


## 7 · RoBERTa pretrained → embedding Excel

`last_hidden_state[:,0,:]` adalah CLS tiapwindow; mean antarwindow menjadi fitur jawaban. **Output:**seluruh jawaban×768 dan setiapCLSwindow→`02_Vektor_Jawaban.xlsx`. Ini SEBELUM fine-tuning; bukan nilai siswa dan bukan similarity yang otomatis terkalibrasi. Semua koordinat tersimpan, preview hanya sedikitkolom.


In [ ]:
%%aes
# Menyimpan vektor RoBERTa sebelum latihan sebagai pembanding untuk hasil sesudah fine-tuning.
def roberta_pretrained_vectors(bundle, cfg, journal):
    device = device_name(cfg)
    model, tok = encoder(cfg, 'roberta').to(device), tokenizer(cfg, 'roberta')
    # Mode pembacaan: dropout dinonaktifkan agar ekstraksi vektor dan prediksi konsisten.
    model.eval()
    vectors, cls_vectors, window_ids = [], [], []
    # Tidak menghitung gradien untuk tahap pembacaan ini; bukan langkah latihan.
    with torch.no_grad():
        for index, text in enumerate(bundle.texts):
            windows, mapping, stats = batch_windows([text], tok, device, cfg['roberta_max_length'], cfg['roberta_overlap'])
            # [:,0,:] adalah hidden state token CLS, bukan seluruh token dirata-rata.
            cls = model(**windows).last_hidden_state[:, 0, :].float()
            vectors.append(mean_by_essay(cls, mapping, 1).cpu().numpy())
            cls_vectors.append(cls.cpu().numpy())
            window_ids.extend(f'{bundle.rows[index]["row_id"]}_window_{j}' for j in range(len(cls)))
            if (index+1) % 25 == 0:
                print(f'RoBERTa sebelum pelatihan: vektor {index+1} dari {len(bundle.rows)} jawaban selesai.', flush=True)
    values = np.concatenate(vectors)
    ids = [row['row_id'] for row in bundle.rows]
    journal.save('05_roberta_pretrained', {'Vektor_jawaban_768': vector_frame(values, ids),
                                          'CLS_tiap_window': vector_frame(np.concatenate(cls_vectors), window_ids)},
                 'Embedding RoBERTa SEBELUM fine-tuning; seluruh koordinat', source=cfg['roberta'],
                 pooling='mean window CLS; bukan sentence similarity yang sudah dikalibrasi')
    del model, tok
    release()
    return values

if RUN_MODELS:
    roberta_before=roberta_pretrained_vectors(bundle,cfg,journal)
    print('RoBERTa: setiap jawaban diubah menjadi vektor berisi 768 angka, sebelum pelatihan dengan nilai guru.')
    show(vector_frame(roberta_before,[r['row_id'] for r in bundle.rows]),'Vektor jawaban RoBERTa sebelum pelatihan')


## 8 · BGE-M3 frozen → embedding Excel

**Output:**seluruh jawaban×1024→`02_Vektor_Jawaban.xlsx`. Dense CLS+L2 mengikuti p14§3.2, `e=norm(H[0])` (PDF gabungan 3135). Vektor belum menjadi skor; head belajar memetakannya. Tidak memakai sparse/ColBERT atau melatih contrastive retrieval ulang.


In [ ]:
%%aes
# Menghitung vektor BGE-M3 sekali karena encodernya frozen; vektor dipakai bersama oleh S2 dan S3.
def bge_features(bundle, cfg, output):
    key = hashlib.sha256(json.dumps({'data': bundle.source_sha256, 'model': cfg['bge'],
                                     'texts': hashlib.sha256(json.dumps(bundle.texts, ensure_ascii=False).encode()).hexdigest(),
                                     'max_length': cfg['bge_max_length'], 'pooling': 'CLS_L2',
                                     'cleaning': 'NFC_controls_only'}, sort_keys=True).encode()).hexdigest()[:20]
    cache = Path(cfg['output'])/'cache'/f'bge_{key}.npz'
    metadata = cache.with_suffix('.json')
    if cache.exists() and metadata.exists():
        proof = json.loads(metadata.read_text())
        if proof['file_sha256'] != hashlib.sha256(cache.read_bytes()).hexdigest():
            raise ValueError('Cache BGE berubah; tidak dipakai.')
        with np.load(cache, allow_pickle=False) as data:
            values = data['features']
        if values.shape != (len(bundle.rows), 1024) or not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values, axis=1), 1, atol=1e-4):
            raise ValueError('Cache BGE shape/normalisasi tidak sah.')
        dump(Path(output)/'bge_encoding.json', {**proof, 'reused': True})
        return torch.from_numpy(values)
    device = device_name(cfg)
    model = encoder(cfg, 'bge').to(device)
    # Mode pembacaan: dropout dinonaktifkan agar ekstraksi vektor dan prediksi konsisten.
    model.eval()
    tok = tokenizer(cfg, 'bge')
    lengths = [len(tok.encode(t, add_special_tokens=True)) for t in bundle.texts]
    output_vectors = []
    # Tidak menghitung gradien untuk tahap pembacaan ini; bukan langkah latihan.
    with torch.no_grad():
        for start in range(0, len(bundle.rows), cfg['bge_batch']):
            batch = tok(bundle.texts[start:start+cfg['bge_batch']], padding=True, truncation=True,
                        max_length=cfg['bge_max_length'], return_tensors='pt')
            batch = {k: v.to(device) for k, v in batch.items() if k in ('input_ids', 'attention_mask')}
            cls = model(**batch).last_hidden_state[:, 0, :].float()
            # Normalkan panjang vektor BGE dengan L2; ini mengatur vektor, bukan menghitung skor siswa.
            output_vectors.append(torch.nn.functional.normalize(cls, dim=1).cpu().numpy())
            processed = min(start+cfg['bge_batch'], len(bundle.rows))
            if processed == len(bundle.rows) or processed % 25 < cfg['bge_batch']:
                print(f'BGE-M3: vektor {processed} dari {len(bundle.rows)} jawaban selesai.', flush=True)
    values = np.concatenate(output_vectors).astype(np.float32)
    if values.shape != (len(bundle.rows), 1024) or not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values, axis=1), 1, atol=1e-4):
        raise RuntimeError('Encoder BGE menghasilkan fitur tidak sah; tidak disimpan.')
    assert not any(p.requires_grad for p in model.parameters())
    cache.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cache, features=values)
    proof = {'model': cfg['bge'], 'shape': list(values.shape), 'frozen': True, 'pooling': 'last_hidden_state CLS then L2',
             'truncated_answers': sum(n > cfg['bge_max_length'] for n in lengths),
             'max_length': cfg['bge_max_length'], 'feature_source': 'response_text only',
             'file_sha256': hashlib.sha256(cache.read_bytes()).hexdigest()}
    dump(metadata, proof)
    dump(Path(output)/'bge_encoding.json', {**proof, 'reused': False})
    del model, tok
    release()
    return torch.from_numpy(values)

if RUN_MODELS:
    bge=bge_features(bundle,cfg,output)
    journal.save('06_bge_frozen',{'Vektor_jawaban_1024':vector_frame(bge.numpy(),[r['row_id'] for r in bundle.rows]),
                                'Encoding':pd.DataFrame([json.loads((output/'bge_encoding.json').read_text())])},
                 'BGE dense frozen; semua koordinat',source=cfg['bge'],pooling='CLS then L2')
    print('BGE-M3: setiap jawaban diubah menjadi vektor berisi 1024 angka. Bobot encoder tetap selama pelatihan.')
    show(vector_frame(bge.numpy(),[r['row_id'] for r in bundle.rows]),'Vektor jawaban BGE-M3')


## 9 · Vektor token/kata/kalimat satu jawaban → Excel

Di tahap ini kita melihat satu jawaban contoh dengan kedua model. **Vektor sebelum konteks** berasal dari tabel embedding model. **Vektor setelah konteks** sudah dipengaruhi posisi dan kata di sekitarnya. Untuk satu kata yang terpecah menjadi beberapa token, vektornya dirata-ratakan dari potongan tersebut. Token yang terbaca di beberapa window dirata-ratakan dahulu.

Setiap baris tabel pertama adalah satu kata; hanya lima komponen vektor ditampilkan di layar. Seluruh angka tersedia di **03_Vektor_Kata_dan_Kalimat.xlsx**. Tabel berikutnya membandingkan dua kemunculan kata yang sama di model yang sama, dengan cara membaca di atas tabel.

Kalimat dibaca secara terpisah dan diwakili token CLS. Kata di luar batas input ditandai tidak tercakup; sel kosong tidak berarti vektor nol. Pemisahan kalimat memakai tanda baca sederhana sehingga bisa keliru pada singkatan. Jangan membandingkan cosine langsung antara ruang RoBERTa 768 angka dan BGE-M3 1024 angka.


In [ ]:
%%aes
WORD = re.compile(r"\b\w+(?:[-’']\w+)*\b", re.UNICODE)
def sentence_spans(text):
    """Segmentasi aturan sederhana, bukan model sintaksis; singkatan bisa terpisah."""
    result = []
    for match in re.finditer(r'[^.!?\n]+(?:[.!?]+|$)|[^.!?\n]+(?=\n)', text):
        start, end = match.span()
        while start < end and text[start].isspace():
            start += 1
        if start < end:
            result.append((start, end))
    return result


# Mengambil vektor lookup awal dan vektor setelah membaca konteks, berikut posisi token di teks.
def token_vectors(text, model, slow, fast, role, cfg, device):
    # Artifact fast RoBERTa mempunyai postprocessor berbeda. Gunakan fast hanya
    # untuk offset TANPA specials, lalu builder slow yang dipakai training.
    aligned = fast(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = slow.encode(text, add_special_tokens=False)
    if list(aligned['input_ids']) != ids:
        raise ValueError('ID fast/slow berbeda; penyelarasan kata tidak boleh ditebak.')
    offsets = [tuple(x) for x in aligned['offset_mapping']]
    if len(offsets) != len(ids):
        raise ValueError('Jumlah offset tidak cocok token.')
    maximum = cfg['roberta_max_length'] if role == 'roberta' else cfg['bge_max_length']
    capacity = maximum-slow.num_special_tokens_to_add(False)
    starts = [0]
    if role == 'roberta':
        while starts[-1]+capacity < len(ids):
            starts.append(starts[-1]+capacity-cfg['roberta_overlap'])
    hidden = model.config.hidden_size
    sums = np.zeros((len(ids), hidden), dtype=np.float32)
    counts = np.zeros(len(ids), dtype=np.int64)
    initial = np.full_like(sums, np.nan)
    cls_vectors = []
    # Mode pembacaan: dropout dinonaktifkan agar ekstraksi vektor dan prediksi konsisten.
    model.eval()
    # Tidak menghitung gradien untuk tahap pembacaan ini; bukan langkah latihan.
    with torch.no_grad():
        for start in starts:
            content = ids[start:start+capacity]
            prepared = slow.build_inputs_with_special_tokens(content)
            # Fast XLM-R menolak get_special_tokens_mask(...False). Tandai
            # posisi konten melalui builder yang sama; sentinel tidak masuk model.
            template = slow.build_inputs_with_special_tokens([-1]*len(content))
            positions = [i for i, v in enumerate(template) if v == -1]
            if len(positions) != len(content) or len(prepared) > maximum or [prepared[i] for i in positions] != content:
                raise ValueError('Builder special token tidak konsisten.')
            tensor = torch.tensor([prepared], dtype=torch.long, device=device)
            output = model(input_ids=tensor, attention_mask=torch.ones_like(tensor)).last_hidden_state
            cls_vectors.append(output[0, 0].float().cpu().numpy())
            stop = start+len(content)
            sums[start:stop] += output[0, positions].float().cpu().numpy()
            counts[start:stop] += 1
            lookup = model.get_input_embeddings()(tensor[:, positions])
            initial[start:stop] = lookup[0].float().cpu().numpy()
    contextual = np.full_like(sums, np.nan)
    np.divide(sums, counts[:, None], out=contextual, where=counts[:, None] > 0)
    sentence_cls = np.mean(cls_vectors, axis=0)
    if role == 'bge':
        sentence_cls = normalize(torch.from_numpy(sentence_cls)[None], dim=1)[0].numpy()
    return {'ids': ids, 'offsets': offsets, 'initial': initial, 'contextual': contextual,
            'counts': counts, 'starts': starts, 'capacity': capacity, 'cls': sentence_cls}


# Merata-ratakan potongan subword yang membentuk satu kata agar perbandingan bisa dibaca per kata.
def word_vectors(text, trace):
    rows, initial, contextual = [], [], []
    for word in WORD.finditer(text):
        pieces = [i for i, (start, end) in enumerate(trace['offsets'])
                  if end > start and start < word.end() and end > word.start()]
        covered = bool(pieces) and all(trace['counts'][i] > 0 for i in pieces)
        dim = trace['initial'].shape[1]
        before = trace['initial'][pieces].mean(axis=0) if covered else np.full(dim, np.nan, dtype=np.float32)
        after = trace['contextual'][pieces].mean(axis=0) if covered else np.full(dim, np.nan, dtype=np.float32)
        rows.append({'word_index': len(rows), 'word': word.group(), 'start': word.start(), 'end': word.end(),
                     'token_indices': pieces, 'token_ids': [trace['ids'][i] for i in pieces],
                     'covered': covered, 'initial_first5': before[:5].tolist() if covered else None,
                     'contextual_first5': after[:5].tolist() if covered else None})
        initial.append(before)
        contextual.append(after)
    dim = trace['initial'].shape[1]
    return rows, np.asarray(initial, dtype=np.float32).reshape(-1, dim), np.asarray(contextual, dtype=np.float32).reshape(-1, dim)


def cosine(a, b):
    denominator = np.linalg.norm(a)*np.linalg.norm(b)
    return float(np.dot(a, b)/denominator) if denominator > 0 else None


# Membandingkan dua kemunculan kata yang sama di model yang sama, sebelum dan sesudah konteks.
def compare_repetitions(rows, initial, contextual, focus=''):
    available = [r for r in rows if r['covered']]
    counts = Counter(r['word'].casefold() for r in available)
    target = focus.casefold().strip() if focus.strip() else next((w for w, n in counts.most_common() if n > 1), '')
    candidates = [r for r in available if r['word'].casefold() == target]
    results = []
    # Hindari tabel kuadratik pada kata sangat sering; semua vektor tetap disimpan.
    for left, right in itertools.islice(itertools.combinations(candidates, 2), 30):
        a, b = left['word_index'], right['word_index']
        results.append({'word': target, 'occurrence_indices': [a, b],
                        'same_token_ids': left['token_ids'] == right['token_ids'],
                        'initial_cosine': cosine(initial[a], initial[b]),
                        'initial_max_abs_difference': float(np.max(np.abs(initial[a]-initial[b]))),
                        'contextual_cosine': cosine(contextual[a], contextual[b]),
                        'contextual_max_abs_difference': float(np.max(np.abs(contextual[a]-contextual[b]))),
                        'contextual_allclose_1e-6': bool(np.allclose(contextual[a], contextual[b], rtol=0, atol=1e-6))})
    return results


def vector_lab(text, cfg, output):
    text = clean(text)
    if not text.strip():
        raise ValueError('Jawaban untuk analisis vektor kosong.')
    output = Path(output)
    destination = output/'vectors'/cfg['vector_source']
    destination.mkdir(parents=True, exist_ok=True)
    results = {}
    for role in ('roberta', 'bge'):
        source = cfg[role]['checkpoint']+'@'+cfg[role]['revision']
        if role == 'roberta' and cfg['vector_source'] != 'pretrained':
            report = json.loads((output/'final_report.json').read_text())
            checkpoint = output/cfg['vector_source']/'final.pt'
            if digest(checkpoint) != report['checkpoints'][cfg['vector_source']]:
                raise ValueError('Checkpoint vektor berbeda dari model final.')
            wrapper, _ = load_model(checkpoint, cfg)
            model = wrapper.roberta
            del wrapper
            source = cfg['vector_source']+' fine-tuned; sha256='+digest(checkpoint)
        else:
            model = encoder(cfg, role)
        device = device_name(cfg)
        model.to(device).eval()
        slow, fast = tokenizer(cfg, role), tokenizer(cfg, role, fast=True)
        trace = token_vectors(text, model, slow, fast, role, cfg, device)
        words, before, after = word_vectors(text, trace)
        sentences, sentence_vectors = [], []
        for start, end in sentence_spans(text):
            part = token_vectors(text[start:end], model, slow, fast, role, cfg, device)
            sentence_vectors.append(part['cls'])
            sentences.append({'sentence_index': len(sentences), 'text': text[start:end], 'start': start, 'end': end,
                              'covered_tokens': int(np.count_nonzero(part['counts'])), 'total_tokens': len(part['ids']),
                              'first5': part['cls'][:5].tolist()})
        vectors = np.asarray(sentence_vectors, dtype=np.float32).reshape(-1, model.config.hidden_size)
        np.savez_compressed(destination/f'{role}_vectors.npz', token_initial=trace['initial'],
                            token_contextual=trace['contextual'], token_ids=np.asarray(trace['ids']),
                            token_offsets=np.asarray(trace['offsets']).reshape(-1, 2), token_coverage=trace['counts'],
                            word_initial=before, word_contextual=after, sentence_cls=vectors)
        token_rows = [{'token_index': i, 'token': slow.convert_ids_to_tokens([identifier])[0], 'token_id': identifier,
                       'start': trace['offsets'][i][0], 'end': trace['offsets'][i][1],
                       'window_coverage': int(trace['counts'][i])} for i, identifier in enumerate(trace['ids'])]
        pd.DataFrame(words).to_csv(destination/f'{role}_words.csv', index=False)
        pd.DataFrame(sentences).to_csv(destination/f'{role}_sentences.csv', index=False)
        pd.DataFrame(token_rows).to_csv(destination/f'{role}_tokens.csv', index=False)
        item = {'source': source, 'dimension': model.config.hidden_size, 'words': words, 'sentences': sentences,
                'tokens': token_rows, 'word_shape': list(after.shape), 'sentence_shape': list(vectors.shape),
                'uncovered_tokens': int(np.count_nonzero(trace['counts'] == 0)), 'window_starts': trace['starts'],
                'repeated_words': compare_repetitions(words, before, after, cfg.get('focus_word', '')),
                'word_method': 'mean subword vectors overlapping each word; contextual overlap windows averaged per token first',
                'sentence_method': 'encode each sentence independently; mean window CLS' if role == 'roberta' else 'encode each sentence independently; CLS then L2; truncate at configured maximum',
                'npz': str(destination/f'{role}_vectors.npz')}
        dump(destination/f'{role}_metadata.json', item)
        results[role] = item
        label = 'RoBERTa' if role == 'roberta' else 'BGE-M3'
        if role == 'roberta':
            label += ' sebelum latihan' if cfg['vector_source']=='pretrained' else f' {cfg["vector_source"]} setelah latihan'
        if role == 'roberta' or cfg['vector_source']=='pretrained':
            print(f'{label}: vektor untuk {len(words)} kata dan {len(sentences)} kalimat selesai. Setiap vektor berisi {model.config.hidden_size} angka.', flush=True)
            if item['uncovered_tokens']:
                print(f'Catatan: {item["uncovered_tokens"]} token melewati batas input dan tidak mendapat vektor.', flush=True)
        del model, slow, fast
        release()
    results['interpretation'] = 'Bandingkan kemunculan dalam encoder yang sama. RoBERTa 768 dan BGE 1024 mempunyai ruang koordinat berbeda: jangan menghitung cosine langsung antarmodel. Same token ID => lookup sama pada checkpoint yang sama; konteks/posisi dapat mengubah hidden state. Kata ejaan sama bisa mempunyai ID subword berbeda (spasi/case).'
    dump(destination/'report.json', results)
    return results
def export_word_sentence_vectors(result, journal, name):
    sheets = {}
    for role in ('roberta', 'bge'):
        report = result[role]
        with np.load(report['npz'], allow_pickle=False) as arrays:
            for key, id_key, metadata in [('token_initial','token_index',report['tokens']),
                                          ('token_contextual','token_index',report['tokens']),
                                          ('word_initial','word_index',report['words']),
                                          ('word_contextual','word_index',report['words']),
                                          ('sentence_cls','sentence_index',report['sentences'])]:
                values = arrays[key]
                frame = vector_frame(values, [row[id_key] for row in metadata])
                label_key = 'token' if key.startswith('token') else 'word' if key.startswith('word') else 'text'
                frame.insert(1, label_key, [row[label_key] for row in metadata])
                if key.startswith('word'):
                    frame.insert(2, 'covered', [row['covered'] for row in metadata])
                sheets[role+'_'+key] = frame
        sheets[role+'_metadata_token'] = pd.DataFrame(report['tokens'])
        sheets[role+'_metadata_word'] = pd.DataFrame(report['words'])
        sheets[role+'_metadata_sentence'] = pd.DataFrame(report['sentences'])
        sheets[role+'_kata_berulang'] = pd.DataFrame(report['repeated_words'])
    journal.save(name, sheets, 'Vektor token/kata/kalimat LENGKAP untuk SATU jawaban',
                 source={r: result[r]['source'] for r in ('roberta','bge')},
                 methods={r:{k:result[r][k] for k in ('word_method','sentence_method','uncovered_tokens','window_starts')} for r in ('roberta','bge')},
                 interpretation=result['interpretation'],
                 missing_vectors='Sel kosong berarti NaN/tidak tercakup; lihat kolom covered/coverage, bukan vektor nol.')

if RUN_WORD_VECTORS:
    before_words=vector_lab(trace_text,{**cfg,'vector_source':'pretrained'},output)
    export_word_sentence_vectors(before_words,journal,'07_vektor_kata_kalimat_pretrained')
    for role in ('roberta','bge'):
        word_rows=before_words[role]['words']
        preview=pd.DataFrame([{'word_index':w['word_index'],'word':w['word'],'covered':w['covered'],
                              **{f'v_{i:04d}':w['contextual_first5'][i] if w['contextual_first5'] is not None else None for i in range(5)}} for w in word_rows])
        show(preview,('RoBERTa' if role=='roberta' else 'BGE-M3')+' — contoh vektor kata')
        from aes.teacher_score.presentation import repeated_word_rows, REPEATED_COLUMNS
        pairs=list(repeated_word_rows(before_words[role]['repeated_words'],role))
        show(pd.DataFrame(pairs,columns=REPEATED_COLUMNS),
             'Perbandingan kata yang berulang',columns=9)


## 10 · Model S1/S2/S3 dan cara menghasilkan skor

**S1** memakai 768 angka RoBERTa untuk menghasilkan satu skor. **S2** memakai 1024 angka BGE-M3, lalu MLP, yaitu jaringan kecil untuk mempelajari pemetaan vektor menjadi skor. **S3** menyambung 768 angka RoBERTa dengan 256 angka hasil proyeksi BGE-M3, sehingga vektor gabungannya berisi 1024 angka.

**Encoder** adalah bagian yang membaca teks dan menghasilkan vektor. **Head**, atau lapisan prediksi, mengubah vektor menjadi skor. Fine-tuned berarti bobot encoder ikut diperbarui saat latihan; frozen berarti bobot encoder tetap. Fungsi `forward` di cell benar-benar dipakai saat pelatihan. Lapisan prediksi yang baru dibuat belum menghasilkan skor yang layak digunakan.

p10 mendukung konsep menggabungkan fitur, tetapi memakai RoBERTa+fitur manual, bukan replikasi RoBERTa+BGE ini. BGE dalam unggahan fullyfine-tuned; di sini S2/S3 frozen sesuai rancangan sebelumnya, head/proyeksi tetapbelajar. **Output spesifikasi:**`05_Riwayat_Pelatihan.xlsx`, jelas diberi statusDESIGN, bukan angka hasilmodel.


In [ ]:
%%aes
# Satu head satu nilai; Engine menyiapkan tensor dan mengembalikan satuan0–100.
# Menyusun encoder dan lapisan prediksi untuk S1, S2, atau S3; ketiganya menghasilkan satu skor per jawaban.
class SingleScoreModel(nn.Module):
    def __init__(self, scenario, roberta=None, dropout=.1):
        super().__init__()
        if scenario not in ('S1', 'S2', 'S3'):
            raise ValueError('Skenario harus S1, S2, atau S3')
        if scenario != 'S2' and roberta is None:
            raise ValueError('RoBERTa diperlukan oleh S1/S3.')
        # Simpan skenario dan encoder RoBERTa. S2 memakai vektor BGE yang sudah dihitung, jadi tidak memuat RoBERTa.
        self.scenario, self.roberta = scenario, roberta
        if roberta is not None:
            # Fine-tuning: bobot RoBERTa boleh berubah ketika optimizer menjalankan langkah latihan.
            roberta.requires_grad_(True)
        self.dropout = nn.Dropout(dropout)
        if scenario == 'S1':
            # S1: 768 koordinat RoBERTa dipetakan ke satu prediksi kontinu.
            self.head = nn.Linear(768, 1)
        elif scenario == 'S2':
            # S2: MLP belajar memberi skor di atas fitur BGE yang tetap.
            self.head = nn.Sequential(nn.Linear(1024, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1))
        else:
            # S3: padatkan 1024 angka BGE menjadi 256 angka sebelum digabung.
            self.projection = nn.Sequential(nn.Linear(1024, 256), nn.LayerNorm(256))
            self.head = nn.Linear(1024, 1)

    def forward(self, windows=None, mapping=None, n_essays=None, bge=None, return_features=False):
        features = {}
        if self.scenario != 'S2':
            # Baca token pada setiap window, lalu ambil vektor token pertama (CLS) sebagai ringkasannya.
            cls = self.roberta(**windows).last_hidden_state[:, 0, :]
            # Rata-ratakan vektor CLS semua window milik jawaban yang sama.
            essay = mean_by_essay(cls, mapping, n_essays)
            features.update(roberta=essay, roberta_window_cls=cls)
        if self.scenario == 'S1':
            # Lapisan prediksi memetakan vektor menjadi satu angka; saat latihan angka ini masih memakai skala target 0–1.
            prediction = self.head(self.dropout(essay))
        elif self.scenario == 'S2':
            if bge is None or bge.ndim != 2 or bge.shape[1] != 1024:
                raise ValueError('BGE memerlukan [N,1024]')
            features['bge'] = bge
            # Lapisan prediksi memetakan vektor menjadi satu angka; saat latihan angka ini masih memakai skala target 0–1.
            prediction = self.head(bge)
        else:
            if bge is None or bge.shape != (n_essays, 1024):
                raise ValueError('BGE memerlukan [N,1024]')
            features['bge'] = bge
            # Proyeksi BGE ikut belajar, walaupun bobot encoder BGE tetap frozen.
            features['projection'] = self.projection(bge)
            # Sambung 768 angka RoBERTa dan 256 angka proyeksi BGE menjadi vektor fusion 1024 angka.
            features['concat'] = torch.cat([essay, features['projection']], dim=1)
            # Lapisan prediksi memetakan vektor menjadi satu angka; saat latihan angka ini masih memakai skala target 0–1.
            prediction = self.head(self.dropout(features['concat']))
        if prediction.ndim != 2 or prediction.shape[1] != 1:
            raise ValueError('Dataset satu skor membutuhkan output [N,1].')
        return (prediction, features) if return_features else prediction


# Menghubungkan teks, tokenizer, dan model. Saat prediksi, hasil dikembalikan ke skala nilai guru.
class Engine:
    def __init__(self, model, texts, bge, cfg, device, roberta_tokenizer=None):
        self.model, self.texts, self.bge, self.cfg, self.device = model, texts, bge, cfg, device
        self.tokenizer = roberta_tokenizer or (tokenizer(cfg, 'roberta') if model.scenario != 'S2' else None)

    def forward(self, indices, return_features=False):
        ids = [int(i) for i in indices]
        features = self.bge[ids].to(self.device) if self.model.scenario != 'S1' else None
        if self.model.scenario == 'S2':
            return self.model(bge=features, return_features=return_features)
        # Pecah jawaban panjang menjadi window; mapping menghubungkan setiap window dengan jawaban asalnya.
        windows, mapping, _ = batch_windows([self.texts[i] for i in ids], self.tokenizer,
                                             self.device, self.cfg['roberta_max_length'], self.cfg['roberta_overlap'])
        return self.model(windows=windows, mapping=mapping, n_essays=len(ids), bge=features,
                          return_features=return_features)

    def predict(self, indices):
        # Mode prediksi: nonaktifkan dropout. Pembacaan di bawah memakai no_grad.
        self.model.eval()
        values = []
        # Tidak menghitung gradien untuk tahap pembacaan ini; bukan langkah latihan.
        with torch.no_grad():
            for start in range(0, len(indices), self.cfg['micro_batch']):
                ids = indices[start:start+self.cfg['micro_batch']]
                values.append(self.forward(ids).reshape(-1).float().cpu().numpy())
        scaled = np.concatenate(values)
        # Kembalikan skala asli sebelum RMSE/Pearson maupun tampilan prediksi.
        # Kembalikan prediksi ke skala nilai guru sebelum ditampilkan atau dihitung RMSE/Pearson.
        return scaled*(self.cfg['score_max']-self.cfg['score_min'])+self.cfg['score_min']


In [ ]:
%%aes
# Model baru pada setiapfold/refit, bukan meneruskan bobot foldlain.
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Membuat model baru agar bobot hasil fold sebelumnya tidak terbawa ke fold berikutnya.
def make_model(scenario, cfg):
    # S1/S3 memuat RoBERTa; S2 hanya membuat lapisan prediksi di atas vektor BGE frozen.
    return SingleScoreModel(scenario, encoder(cfg, 'roberta') if scenario != 'S2' else None,
                            dropout=cfg['dropout'])


In [ ]:
%%aes
specifications=pd.DataFrame([
    {'scenario':'S1','encoder':'RoBERTa: bobot ikut dilatih','vector':768,'head':'Linear: 768 angka → 1 skor','output':1},
    {'scenario':'S2','encoder':'BGE-M3: bobot tetap','vector':1024,'head':'MLP: 1024 → 256 → 1 skor','output':1},
    {'scenario':'S3','encoder':'RoBERTa ikut dilatih; BGE-M3 tetap','vector':1024,'head':'Gabungan RoBERTa 768 + proyeksi BGE 256 → 1 skor','output':1}])
journal.save('08_arsitektur',{'Spesifikasi':specifications},'Spesifikasi model, bukan prediksi',status='DESIGN / CONFIGURATION')
show(specifications,'Arsitektur dan bagian yang belajar')


## 11 · Empat metrik, loss, optimizer dan batch

RMSE/Pearson memakai prediksi mentah 0–100. QWK/AC2 kuadratik memakai salinan clip 0–100 lalu integerterdekat(tie.5 kebawah),101 kategori dipertahankan. Undefined=kosong+alasan. AC2 sensitif kategoridan bisa tinggi saat QWK buruk; baca bersama metrik lain(m06). Rumus AC2 juga diverifikasi dengan sourceirrCAC; bab rumus tidak tersedia dalam cuplikan bukum 08.

`fit` di bawah adalah fungsi latihan sebenarnya. RoBERTaLR 2e−5; head/proyeksi 1e−3; AdamW,MSE,warmup 10%,clipgrad 1. Microbatch 1 diakumulasi menjadi efektif 8. Target 0–1 hanya untukloss, RMSE tetap poinasli. **Output sesudahtraining:**riwayatepoch/loss/metrik/epochpilihan→Excel tiap fold/refit.

Saat berjalan, **epoch** berarti satu putaran data latih. **Loss latih (MSE, target 0–1)** adalah kesalahan yang diperkecil saat model belajar, dengan nilai guru dibagi 100. **RMSE validasi** sudah dalam poin skor asli 0–100. Keduanya memakai skala berbeda, sehingga angkanya tidak dibandingkan langsung.


In [ ]:
%%aes
# Agreement kuadratik Gwet, bukan sekadar korelasi.
def gwet_ac2(teacher, predicted, categories):
    a=np.asarray(teacher);b=np.asarray(predicted);c=np.asarray(categories,dtype=float)
    if len(a)!=len(b) or len(a)<2: raise ValueError('AC2 perlu minimal dua pasangan')
    if not np.isin(a,c).all() or not np.isin(b,c).all(): raise ValueError('Kategori AC2 tidak sah')
    q=len(c); weights=1-((c[:,None]-c[None,:])/(c.max()-c.min()))**2
    ai=np.searchsorted(c,a); bi=np.searchsorted(c,b)
    pa=weights[ai,bi].mean()
    pi=(np.bincount(ai,minlength=q)+np.bincount(bi,minlength=q))/(2*len(a))
    pe=weights.sum()*np.sum(pi*(1-pi))/(q*(q-1))
    return float((pa-pe)/(1-pe))


In [ ]:
%%aes
# Empat metrik tanpa menimpa prediksi mentah.
# Membandingkan nilai guru dan prediksi: RMSE/Pearson memakai skor mentah; QWK/AC2 memakai salinan kategori.
def metrics(teacher, prediction, cfg):
    y, p = np.asarray(teacher, dtype=float), np.asarray(prediction, dtype=float)
    if y.ndim != 1 or y.shape != p.shape or len(y) == 0:
        raise ValueError('Metrik satu skor memerlukan dua vektor [N] yang sama dan tidak kosong.')
    if not np.isfinite(y).all() or not np.isfinite(p).all():
        raise ValueError('Label/prediksi tidak finite.')
    cats = np.arange(cfg['score_min'], cfg['score_max']+1)
    if not np.isin(y, cats).all():
        raise ValueError('Skor guru harus kategori integer pada rentang yang dinyatakan.')
    # Buat SALINAN skor sah untuk agreement; prediksi mentah tetap utuh.
    clipped = np.clip(p, cfg['score_min'], cfg['score_max'])
    # Tie .5 memilih kategori lebih rendah, sama dengan kontrak konversi proyek.
    ordinal = cats[np.abs(clipped[:, None]-cats[None, :]).argmin(axis=1)]
    result = {'RMSE': float(np.sqrt(np.mean((y-p)**2))), 'Pearson': None,
              'QWK': None, 'AC2': None, 'undefined_reasons': {},
              'out_of_range_fraction': float(np.mean((p < cats[0]) | (p > cats[-1])))}
    if len(y) > 1 and np.ptp(y) > 0 and np.ptp(p) > 0:
        result['Pearson'] = float(pearsonr(y, p).statistic)
    else:
        result['undefined_reasons']['Pearson'] = 'Perlu minimal dua pasangan dan kedua vektor tidak konstan.'
    if len(y) > 1:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            # QWK memakai jarak kategori kuadratik, bukan vektor embedding.
            qwk = cohen_kappa_score(y, ordinal, labels=cats, weights='quadratic')
        if np.isfinite(qwk):
            result['QWK'] = float(qwk)
        else:
            result['undefined_reasons']['QWK'] = 'Expected disagreement nol; undefined, bukan nol.'
        # AC2 memakai kategori yang sama dengan QWK dan koreksi peluang berbeda.
        result['AC2'] = gwet_ac2(y, ordinal, cats)
    else:
        result['undefined_reasons'].update(QWK='Perlu minimal dua pasangan.', AC2='Perlu minimal dua pasangan.')
    return result


In [ ]:
%%aes
# backward menghitung gradien; optimizer.step memperbarui parameter.
# Melatih satu model pada satu pembagian data; RMSE validasi menentukan epoch yang disimpan.
def fit(scenario, training, validation, bundle, bge, cfg, epochs, seed):
    seed_everything(seed)
    device = device_name(cfg)
    # Buat model baru untuk fold ini, lalu tempatkan di GPU atau CPU yang tersedia.
    model = make_model(scenario, cfg).to(device)
    engine = Engine(model, bundle.texts, bge, cfg, device)
    # Pilih bobot encoder RoBERTa yang menerima gradien.
    enc = [p for name, p in model.named_parameters() if name.startswith('roberta.') and p.requires_grad]
    # Pilih bobot head/proyeksi; kecepatannya bisa berbeda dari encoder.
    head = [p for name, p in model.named_parameters() if not name.startswith('roberta.') and p.requires_grad]
    groups = ([{'params': enc, 'lr': cfg['encoder_lr']}] if enc else [])+[
        {'params': head, 'lr': cfg['head_lr']}]
    # AdamW memperbarui kelompok parameter sesuai learning rate masing-masing.
    optimizer = torch.optim.AdamW(groups, weight_decay=cfg['weight_decay'])
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader([int(i) for i in training], batch_size=cfg['effective_batch'],
                        shuffle=True, generator=generator, num_workers=0)
    total_steps = len(loader)*epochs
    warmup = min(int(cfg['warmup_ratio']*total_steps), max(0, total_steps-1))
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup, total_steps)
    use_cuda = device == 'cuda'
    dtype = torch.bfloat16 if use_cuda and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda and dtype == torch.float16)
    selected_parameters = {}
    for name, parameter in model.named_parameters():
        group = name.split('.')[0]
        if parameter.requires_grad and group not in selected_parameters:
            selected_parameters[group] = (name, parameter.detach().reshape(-1)[:32].cpu().clone())
    best_state, best_epoch, best_rmse = None, 0, float('inf')
    history, active_steps, optimizer_steps, skipped_steps = [], 0, 0, 0
    for epoch in range(1, epochs+1):
        model.train()
        loss_total = 0.
        for effective in loader:
            batch = effective.tolist()
            optimizer.zero_grad(set_to_none=True)
            for start in range(0, len(batch), cfg['micro_batch']):
                ids = batch[start:start+cfg['micro_batch']]
                # Ubah nilai guru dari skala 0–100 menjadi 0–1 hanya untuk menghitung loss.
                targets = torch.as_tensor((bundle.y[ids]-cfg['score_min'])/(cfg['score_max']-cfg['score_min']),
                                          dtype=torch.float32, device=device).reshape(-1, 1)
                with torch.autocast(device_type=device, dtype=dtype, enabled=use_cuda):
                    prediction = engine.forward(ids).float()
                    if prediction.shape != targets.shape:
                        raise ValueError('Shape prediksi/target harus [B,1], tanpa broadcasting.')
                    # MSE mengukur selisih kuadrat antara prediksi dan target; optimizer berusaha memperkecilnya.
                    loss = torch.mean((prediction-targets)**2)
                if not torch.isfinite(loss):
                    raise RuntimeError('Loss nonfinite; run dihentikan, tidak diklaim berhasil.')
                # Hitung gradien dan bobot microbatch sesuai ukuran batch efektif aktual.
                scaler.scale(loss*(len(ids)/len(batch))).backward()
                loss_total += loss.item()*len(ids)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['gradient_clip'])
            lr_active = any(g['lr'] > 0 for g in optimizer.param_groups)
            previous_scale = scaler.get_scale()
            # Perbarui bobot; GradScaler bisa melewati langkah yang overflow pada AMP.
            scaler.step(optimizer)
            scaler.update()
            skipped = scaler.is_enabled() and scaler.get_scale() < previous_scale
            if skipped:
                skipped_steps += 1
            else:
                active_steps += int(lr_active)
                scheduler.step()
                optimizer_steps += 1
        row = {'epoch': epoch, 'train_mse_scaled': loss_total/len(training),
               'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
               'effective_batch_target': cfg['effective_batch'], 'micro_batch': cfg['micro_batch']}
        if validation is not None:
            predicted = engine.predict(validation)
            row['validation'] = metrics(bundle.y[validation], predicted, cfg)
            # Simpan epoch dengan RMSE validasi terkecil; data uji akhir tidak ikut memilih epoch.
            if row['validation']['RMSE'] < best_rmse:
                best_rmse, best_epoch = row['validation']['RMSE'], epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            message = f" | RMSE validasi: {row['validation']['RMSE']:.4f} poin"
        else:
            best_epoch = epoch
            message = ' | pelatihan final; data uji belum dibuka'
        history.append(row)
        print(f'{scenario} · epoch {epoch}/{epochs} · loss latih (MSE, target 0–1): {row["train_mse_scaled"]:.6f}'+message, flush=True)
    if active_steps == 0:
        raise RuntimeError('Tidak ada pembaruan dengan LR aktif.')
    named = dict(model.named_parameters())
    changed = {group: not torch.equal(before, named[name].detach().reshape(-1)[:32].cpu())
               for group, (name, before) in selected_parameters.items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    # Mode pembacaan: dropout dinonaktifkan agar ekstraksi vektor dan prediksi konsisten.
    model.eval()
    info = {'scenario': scenario, 'best_epoch': best_epoch, 'epochs_completed': len(history),
            'history': history, 'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
            'parameter_samples_changed': changed, 'BGE_encoder_trained': False,
            'selection': 'minimum validation RMSE in original score units' if validation is not None else 'fixed refit epochs selected from CV'}
    return model, engine, info


## 12 · Pelatihan dan evaluasi tiga skenario

            Model dilatih dengan lima fold pada data development. Epoch terbaik dipilih dari RMSE validasi, kemudian ketiga model dilatih ulang pada development sebelum final test dibuka. Cara ini tetap sama seperti versi sebelumnya.

            Riwayat seluruh fold digabung per model pada **05_Riwayat_Pelatihan.xlsx**. Prediksi dan empat metrik tersedia pada **04_Prediksi_dan_Evaluasi.xlsx**. File rincian per fold tetap disimpan oleh program, tetapi tidak dimasukkan ke paket utama.


In [ ]:
%%aes
# CV semua skenario, refitsemua, lalu finaltest dengan checkpoint terkunci.
def save_model(path, model, cfg, bundle, manifest, info):
    payload = {'scope': SCOPE, 'trained': True, 'scenario': model.scenario,
               'source_sha256': bundle.source_sha256, 'split_sha256': manifest['split_sha256'],
               'config': cfg, 'training': info,
               'state': {k: v.detach().cpu() for k, v in model.state_dict().items()}}
    torch.save(payload, path)


def load_model(path, cfg):
    # Checkpoint hanya dari workflow ini; weights_only membatasi objek pickle.
    payload = torch.load(path, map_location='cpu', weights_only=True)
    if payload.get('scope') != SCOPE or payload.get('trained') is not True:
        raise ValueError('Checkpoint bukan model satu skor guru yang telah dilatih.')
    for key in ('roberta', 'bge', 'score_min', 'score_max', 'roberta_max_length', 'roberta_overlap', 'bge_max_length'):
        if payload['config'][key] != cfg[key]:
            raise ValueError('Metadata checkpoint berbeda: '+key)
    model = make_model(payload['scenario'], cfg)
    model.load_state_dict(payload['state'], strict=True)
    # Mode pembacaan: dropout dinonaktifkan agar ekstraksi vektor dan prediksi konsisten.
    model.eval()
    return model, payload


# Menjalankan lima fold tiap skenario, latihan final, lalu evaluasi data uji setelah pilihan model dikunci.
def train_all(bundle, manifest, bge, cfg, output, observer=None):
    output = Path(output)
    final_path = output/'final_report.json'
    if final_path.exists():
        report = json.loads(final_path.read_text())
        if report['source_sha256'] != bundle.source_sha256 or report['split_sha256'] != manifest['split_sha256']:
            raise ValueError('Hasil lama memakai sumber/split berbeda.')
        for scenario in SCENARIOS:
            if digest(output/scenario/'final.pt') != report['checkpoints'][scenario]:
                raise ValueError('Checkpoint hasil lama berubah.')
            if digest(output/scenario/'test_predictions.csv') != report['prediction_files'][scenario]:
                raise ValueError('Prediksi test tersimpan berubah.')
        print('Memakai hasil tersimpan yang cocok dengan data, pengaturan, dan versi kode. Pelatihan serta evaluasi data uji tidak diulang.', flush=True)
        return {**report, 'reused_completed_run': True}
    cv_rows, selected_epochs, cv_folds = [], {}, {}
    for scenario in SCENARIOS:
        destination = output/scenario
        destination.mkdir(exist_ok=True)
        completed = []
        for fold in manifest['folds']:
            file = destination/f'cv_fold_{fold["fold"]}.json'
            if file.exists():
                result = json.loads(file.read_text())
                if result['split_sha256'] != manifest['split_sha256'] or result['fold'] != fold['fold']:
                    raise ValueError('Cache fold memakai split berbeda.')
                print(f'{scenario}: memakai hasil validasi silang fold {fold["fold"]} yang sudah selesai.', flush=True)
            else:
                print(f'{scenario}: validasi silang, fold {fold["fold"]} dari {cfg["folds"]}. Data uji disimpan untuk evaluasi akhir.', flush=True)
                model, engine, info = fit(scenario, fold['train'], fold['validation'], bundle, bge, cfg,
                                          cfg['epochs'], cfg['seed']+fold['fold'])
                prediction = engine.predict(fold['validation'])
                result = {'fold': fold['fold'], 'split_sha256': manifest['split_sha256'],
                          'training': info, 'metrics': metrics(bundle.y[fold['validation']], prediction, cfg),
                          'validation_indices': fold['validation'], 'prediction': prediction.tolist()}
                dump(file, result)
                del engine, model
                release()
            completed.append(result)
            if observer is not None:
                observer('cv_fold', scenario, result)
            cv_rows.append({'Skenario': scenario, 'Fold': fold['fold'],
                            **{k: result['metrics'][k] for k in ('QWK', 'AC2', 'RMSE', 'Pearson')}})
        cv_folds[scenario] = completed
        selected_epochs[scenario] = int(math.ceil(np.median([r['training']['best_epoch'] for r in completed])))
    pd.DataFrame(cv_rows).to_csv(output/'cv_metrics.csv', index=False)
    selection = {'scope': SCOPE, 'selected_epochs': selected_epochs,
                 'method': 'ceil median best validation-RMSE epoch across development folds',
                 'all_scenarios_locked_before_final_test': True, 'split_sha256': manifest['split_sha256']}
    dump(output/'selection_locked.json', selection)
    # Seluruh refit selesai sebelum membaca prediksi/skor test untuk pelaporan.
    lock = output/'refit_locked.json'
    if lock.exists():
        previous = json.loads(lock.read_text())
        checksums = previous['checkpoints']
        if previous['selection'] != selection or any(digest(output/s/'final.pt') != checksums[s] for s in SCENARIOS):
            raise ValueError('Model/config final yang sudah dikunci berubah; jangan mengulang test untuk tuning.')
    else:
        checksums = {}
        for scenario in SCENARIOS:
            model, engine, info = fit(scenario, manifest['development'], None, bundle, bge, cfg,
                                      selected_epochs[scenario], cfg['seed'])
            path = output/scenario/'final.pt'
            save_model(path, model, cfg, bundle, manifest, info)
            if observer is not None:
                observer('refit', scenario, info)
            checksums[scenario] = digest(path)
            del engine, model
            release()
        dump(lock, {'scope': SCOPE, 'checkpoints': checksums,
                    'selection': selection, 'test_evaluated': False})
    rows, prediction_files = [], {}
    for scenario in SCENARIOS:
        completed_test = output/scenario/'test_result.json'
        if completed_test.exists():
            cached = json.loads(completed_test.read_text())
            if cached['checkpoint_sha256'] != checksums[scenario] or cached['split_sha256'] != manifest['split_sha256'] or digest(output/scenario/'test_predictions.csv') != cached['prediction_sha256']:
                raise ValueError('Hasil test parsial berubah; tidak dievaluasi ulang.')
            rows.append(cached['row'])
            prediction_files[scenario] = cached['prediction_sha256']
            continue
        model, payload = load_model(output/scenario/'final.pt', cfg)
        if payload['source_sha256'] != bundle.source_sha256 or payload['split_sha256'] != manifest['split_sha256']:
            raise ValueError('Checkpoint final berbeda sumber/split.')
        model.to(device_name(cfg))
        engine = Engine(model, bundle.texts, bge, cfg, device_name(cfg))
        prediction = engine.predict(manifest['test'])
        score = metrics(bundle.y[manifest['test']], prediction, cfg)
        rows.append({'Skenario': scenario, **{k: score[k] for k in ('QWK', 'AC2', 'RMSE', 'Pearson')},
                     'Status': 'test held-out; satu skor guru', 'undefined_reasons': score['undefined_reasons'],
                     'out_of_range_fraction': score['out_of_range_fraction']})
        frame = pd.DataFrame({'row_id': [bundle.rows[i]['row_id'] for i in manifest['test']],
                              'skor_guru': bundle.y[manifest['test']], 'prediksi_mentah': prediction,
                              'selisih': prediction-bundle.y[manifest['test']]})
        path = output/scenario/'test_predictions.csv'
        frame.to_csv(path, index=False)
        prediction_files[scenario] = digest(path)
        dump(completed_test, {'row': rows[-1], 'checkpoint_sha256': checksums[scenario],
                              'split_sha256': manifest['split_sha256'], 'prediction_sha256': digest(path)})
        if observer is not None:
            observer('test', scenario, {'metrics': rows[-1], 'predictions': frame.to_dict('records')})
        del engine, model
        release()
    pd.DataFrame([{k: row[k] for k in ('Skenario', 'QWK', 'AC2', 'RMSE', 'Pearson', 'Status')} for row in rows]).to_csv(output/'test_metrics.csv', index=False)
    report = {'scope': SCOPE, 'source_sha256': bundle.source_sha256, 'split_sha256': manifest['split_sha256'],
              'rows': rows, 'checkpoints': checksums, 'prediction_files': prediction_files,
              'selected_epochs': selected_epochs, 'test_rows': manifest['test_rows'],
              'test_evaluations_per_scenario': 1, 'reused_completed_run': False,
              'cv': cv_rows, 'four_trait_labels_available': False}
    dump(final_path, report)
    return report


In [ ]:
%%aes
def ordinal_copy(prediction, cfg):
    categories = np.arange(cfg['score_min'], cfg['score_max']+1)
    return categories[np.abs(np.clip(prediction, categories[0], categories[-1])[:, None]-categories).argmin(axis=1)]


def prediction_frame(bundle, indices, prediction, cfg):
    teacher = bundle.y[indices]
    ordinal = ordinal_copy(np.asarray(prediction), cfg)
    return pd.DataFrame({'row_id': [bundle.rows[i]['row_id'] for i in indices],
                         'skor_guru': teacher, 'prediksi_mentah': prediction,
                         'kategori_QWK_AC2': ordinal, 'selisih_mentah': prediction-teacher,
                         'selisih_kuadrat': (prediction-teacher)**2,
                         'bobot_agreement_kuadratik': 1-((ordinal-teacher)/(cfg['score_max']-cfg['score_min']))**2})


def export_training_event(kind, scenario, result, bundle, cfg, journal):
    if kind == 'cv_fold':
        history = []
        for entry in result['training']['history']:
            history.append({**{k: v for k, v in entry.items() if k != 'validation'},
                            **entry.get('validation', {})})
        predictions = prediction_frame(bundle, result['validation_indices'], np.asarray(result['prediction']), cfg)
        journal.save(f'09_{scenario}_fold_{result["fold"]}', {'Riwayat_epoch': pd.DataFrame(history),
                                                           'Prediksi_validation': predictions,
                                                           'Metrik_validation': pd.DataFrame([result['metrics']])},
                     'CV development; bukan final test', scenario=scenario,
                     selected_epoch=result['training']['best_epoch'], encoder_fine_tuned=scenario != 'S2')
    elif kind == 'refit':
        journal.save(f'10_{scenario}_refit', {'Riwayat_epoch': pd.DataFrame(result['history']),
                                            'Bukti_parameter': pd.DataFrame([result['parameter_samples_changed']])},
                     'Refit seluruh development; test belum dibuka', scenario=scenario,
                     epochs=result['epochs_completed'], BGE_encoder_trained=False)
    elif kind == 'test':
        raw = pd.DataFrame(result['predictions'])
        lookup = {r['row_id']: i for i, r in enumerate(bundle.rows)}
        ids = [lookup[row_id] for row_id in raw['row_id']]
        expanded = prediction_frame(bundle, ids, raw['prediksi_mentah'].to_numpy(), cfg)
        journal.save(f'11_{scenario}_test', {'Prediksi_test': expanded,
                                           'Metrik_test': pd.DataFrame([result['metrics']])},
                     'Final test satu skor guru', scenario=scenario)


def export_completed_training(report, bundle, cfg, output, journal):
    output = Path(output)
    # Juga membangun ulang Excel dari artefak cocok hash ketika run di-resume.
    for scenario in ('S1','S2','S3'):
        for path in sorted((output/scenario).glob('cv_fold_*.json')):
            export_training_event('cv_fold', scenario, json.loads(path.read_text()), bundle, cfg, journal)
        checkpoint = output/scenario/'final.pt'
        if digest(checkpoint) != report['checkpoints'][scenario]:
            raise ValueError('Checkpoint final berbeda dari yang dievaluasi.')
        payload = torch.load(checkpoint, map_location='cpu', weights_only=True)
        export_training_event('refit', scenario, payload['training'], bundle, cfg, journal)
        del payload
        predicted = pd.read_csv(output/scenario/'test_predictions.csv')
        row = next(x for x in report['rows'] if x['Skenario'] == scenario)
        export_training_event('test', scenario, {'metrics': row, 'predictions': predicted.to_dict('records')}, bundle, cfg, journal)
    cv = pd.DataFrame(report['cv'])
    summary = cv.groupby('Skenario')[['QWK','AC2','RMSE','Pearson']].agg(['mean','std','count'])
    summary.columns = [f'{metric}_{stat}' for metric,stat in summary.columns]
    journal.save('12_evaluasi_tiga_skenario', {'CV_semua_fold': cv, 'CV_mean_SD_n': summary.reset_index(),
                                             'TEST_final': pd.DataFrame(report['rows']),
                                             'Epoch_refit': pd.DataFrame([report['selected_epochs']])},
                 'Empat metrik QWK/AC2/RMSE/Pearson; CV dan test terpisah',
                 test_evaluations_per_scenario=report['test_evaluations_per_scenario'])

if RUN_TRAINING:
    if not RUN_MODELS:raise RuntimeError('Aktifkan RUN_MODELS sebelumtraining.')
    def on_training_stage(kind,scenario,result):
        export_training_event(kind,scenario,result,bundle,cfg,journal)
    report=train_all(bundle,manifest,bge,cfg,output,observer=on_training_stage)
    export_completed_training(report,bundle,cfg,output,journal)
    show(pd.DataFrame(report['cv']),'Hasil validasi silang (lima fold)',rows=15)
    final_table=pd.DataFrame(report['rows'])[['Skenario','RMSE','Pearson','QWK','AC2']]
    show(final_table,'Perbandingan model pada data uji',rows=3)
else:
    print('Pelatihan belum dijalankan. Prediksi dan evaluasi belum tersedia.')


## 13 · Prediksi satu jawaban dengan 3 model final

Sama seperti akhir file unggahan: tampilkan prediksi. `NEW_ANSWER`kosong memakai contohheld-out; teks baru tidak memerlukan nilai guru dan tidak melatih ulang. **Output:**3 skormentah,kategorisalinan,out-of-range→`04_Prediksi_dan_Evaluasi.xlsx`.


In [ ]:
%%aes
# Membaca satu jawaban dengan ketiga model final untuk menghasilkan prediksi; tidak melatih model lagi.
def score_answer(text, cfg, output):
    """Prediksi baru tanpa memerlukan nilai guru jawaban yang akan diprediksi."""
    from aes.cleaning import clean
    output = Path(output)
    text = clean(text)
    if not text.strip():
        raise ValueError('Jawaban prediksi kosong.')
    source = hashlib.sha256(text.encode()).hexdigest()
    # Inference tidak mempunyai target/kelompok training; jangan mengisi skor 0.
    sample = Bundle([{'text': text}], np.empty(0), np.empty(0, dtype=int), source, {})
    features = bge_features(sample, cfg, output/'prediction_encoding')
    rows = []
    report = json.loads((output/'final_report.json').read_text())
    for scenario in SCENARIOS:
        path = output/scenario/'final.pt'
        if digest(path) != report['checkpoints'][scenario]:
            raise ValueError('Checkpoint prediksi berbeda dari final yang dievaluasi.')
        model, payload = load_model(path, cfg)
        model.to(device_name(cfg))
        engine = Engine(model, [text], features, cfg, device_name(cfg))
        prediction = float(engine.predict([0])[0])
        cats = np.arange(cfg['score_min'], cfg['score_max']+1)
        rounded = int(cats[np.abs(cats-np.clip(prediction, cats[0], cats[-1])).argmin()])
        rows.append({'Skenario': scenario, 'prediksi_mentah': prediction, 'kategori_salinan': rounded,
                     'di_luar_rentang': not cfg['score_min'] <= prediction <= cfg['score_max'],
                     'target': 'satu skor guru; bukan empat rubrik'})
        del engine, model
        release()
    dump(output/'one_answer_prediction.json', {'rows': rows})
    return rows

if RUN_TRAINING:
    prediction_rows=score_answer(trace_text,cfg,output)
    journal.save('13_prediksi_satu_jawaban',{'Input_privat':pd.DataFrame([{'text':trace_text}]),
                                           'Tiga_skenario':pd.DataFrame(prediction_rows)},'Prediksi checkpointfinal; bukan trainingulang')
    show(pd.DataFrame(prediction_rows)[['Skenario','prediksi_mentah','kategori_salinan','di_luar_rentang']],
         'Prediksi skor satu jawaban (skala 0–100)',columns=4)


## 14 · Lihat vektor setelah model belajar

            Tahap penghitungan tetap mengambil fitur model final setelah evaluasi. Untuk laporan utama, cukup gunakan vektor RoBERTa sebelum/sesudah latihan, BGE-M3, dan gabungan fusion pada **02_Vektor_Jawaban.xlsx**.

            Vektor kata dan kalimat contoh tersedia pada **03_Vektor_Kata_dan_Kalimat.xlsx**. Angka lapisan internal seperti dropout atau aktivasi tidak perlu dilampirkan ke dosen; rincian itu tetap disimpan secara privat oleh program.


In [ ]:
%%aes
# Menyimpan vektor model yang sudah dilatih agar dapat dibandingkan dengan vektor sebelum latihan.
def export_final_features(bundle, bge, cfg, output, journal):
    output = Path(output)
    report = json.loads((output/'final_report.json').read_text())
    for scenario in ('S1','S2','S3'):
        checkpoint = output/scenario/'final.pt'
        if digest(checkpoint) != report['checkpoints'][scenario]:
            raise ValueError('Checkpoint fitur berubah.')
        model, payload = load_model(checkpoint, cfg)
        del payload
        model.to(device_name(cfg)).eval()
        engine = Engine(model, bundle.texts, bge, cfg, device_name(cfg))
        hooks, intermediates = [], {}
        def save_hidden(name):
            def callback(module, args, value):
                intermediates[name] = value.detach().float().cpu().numpy()
            return callback
        # Tangkap keluaran head/proyeksi aktual tanpa mengganti arsitektur.
        if scenario == 'S2':
            for name, layer in [('MLP_linear256',model.head[0]),('MLP_GELU256',model.head[1]),('MLP_dropout_eval',model.head[2])]:
                hooks.append(layer.register_forward_hook(save_hidden(name)))
        if scenario == 'S3':
            hooks.append(model.projection[0].register_forward_hook(save_hidden('proyeksi_linear256')))
        if scenario != 'S2':
            hooks.append(model.dropout.register_forward_hook(save_hidden('dropout_eval')))
        values, window_values, window_ids = {}, [], []
        try:
            # Tidak menghitung gradien untuk tahap pembacaan ini; bukan langkah latihan.
            with torch.no_grad():
                for index, row in enumerate(bundle.rows):
                    prediction, features = engine.forward([index], return_features=True)
                    for name, value in features.items():
                        array = value.float().cpu().numpy()
                        if name == 'roberta_window_cls':
                            window_values.append(array)
                            window_ids.extend(f'{row["row_id"]}_window_{j}' for j in range(len(array)))
                        else:
                            values.setdefault(name, []).append(array)
                    for name, array in intermediates.items():
                        values.setdefault(name, []).append(array)
                    values.setdefault('output_normalized', []).append(prediction.float().cpu().numpy())
        finally:
            for handle in hooks:
                handle.remove()
        ids = [r['row_id'] for r in bundle.rows]
        sheets = {name: vector_frame(np.concatenate(parts), ids) for name, parts in values.items()}
        if window_values:
            sheets['CLS_tiap_window'] = vector_frame(np.concatenate(window_values), window_ids)
        raw = np.concatenate(values['output_normalized']).reshape(-1)*(cfg['score_max']-cfg['score_min'])+cfg['score_min']
        sheets['Prediksi_semua_jawaban'] = pd.DataFrame({'row_id':ids, 'prediksi_mentah':raw})
        journal.save(f'14_{scenario}_fitur_setelah_training', sheets,
                     'Fitur aktual checkpoint final; diekstrak SESUDAH final test, mode eval',
                     source={'scenario':scenario,'checkpoint_sha256':digest(checkpoint)},
                     note='Seluruh jawaban untuk inspeksi fitur; evaluasi penelitian hanya pada TEST_final, bukan tabel semua jawaban.')
        del engine, model
        release()

if RUN_TRAINING and RUN_FINAL_FEATURES:
    print('Mengambil vektor jawaban dari ketiga model yang sudah dilatih...')
    export_final_features(bundle,bge,cfg,output,journal)
    print('Vektor',len(bundle.rows),'jawaban siap disajikan dalam 02_Vektor_Jawaban.xlsx.')
    if RUN_WORD_VECTORS:
        for scenario in ('S1','S3'):
            after_words=vector_lab(trace_text,{**cfg,'vector_source':scenario},output)
            export_word_sentence_vectors(after_words,journal,f'15_{scenario}_vektor_kata_setelah_training')


## 15 · Unduh hasil Excel yang sudah rapi

            Jalankan **satu cell terakhir di bawah**. Colab utama ini langsung menyusun hasil menjadi lima Excel dan mengunduh **Hasil_AES_Ringkas.zip**. Tidak perlu membuka notebook lain atau mengunggah hasil untuk dirapikan lagi.

            Paket akhir **Hasil_AES_Ringkas.zip** berisi:
            1. **Data dan Tokenisasi** — jawaban, nilai guru, pembagian data, serta token.
            2. **Vektor Jawaban** — seluruh komponen vektor per jawaban, sebelum/sesudah latihan dan fusion.
            3. **Vektor Kata dan Kalimat** — satu contoh jawaban dan perbandingan kata berulang.
            4. **Prediksi dan Evaluasi** — buka sheet **Perbandingan Model** terlebih dahulu.
            5. **Riwayat Pelatihan** — fold dan epoch digabung per model.

            Tiap file diawali sheet **Panduan**. Nama kolom sudah dirapikan; semua komponen vektor utama tetap lengkap. Rincian lapisan internal tidak masuk ZIP unduhan.

            Jika proses baru selesai sebagian, paket hanya memuat tahap yang tersedia, tanpa hasil pengganti. Untuk mengunduh ulang di sesi yang sama, cukup jalankan cell terakhir ini; cell ini tidak melatih atau mengevaluasi model ulang.


In [ ]:
# Susun laporan dari hasil sesi ini. Tidak memanggil training atau evaluasi lagi.
aes_cell('', """
from aes.teacher_score.presentation import build_reports
summary=build_reports(journal.folder,output/'hasil_ringkas',verbose=False)
dump(ROOT/'outputs/dataset300_download.json',{
    'archive':summary['archive'],'run_folder':str(output),
    'excel_count':len(summary['files']),'files':[Path(p).name for p in summary['files']]})
""")

# Unduh paket yang baru dibuat, bukan ZIP rincian teknis atau hasil sesi lain.
import zipfile
download=json.loads((ROOT/'outputs/dataset300_download.json').read_text())
with zipfile.ZipFile(download['archive']) as package:
    names=package.namelist()
if sorted(names)!=sorted(download['files']):
    raise RuntimeError('Isi ZIP tidak cocok dengan daftar laporan. Unduhan dihentikan.')
print('Hasil siap diunduh:',Path(download['archive']).name)
print('Isi paket:',len(names),'file Excel yang sudah dirapikan:')
descriptions={
    '01_Data_dan_Tokenisasi.xlsx':'Jawaban, nilai guru, pembagian data, dan token kedua model.',
    '02_Vektor_Jawaban.xlsx':'Embedding jawaban RoBERTa, BGE-M3, dan fusion.',
    '03_Vektor_Kata_dan_Kalimat.xlsx':'Vektor kata/kalimat dan perbandingan kata yang berulang.',
    '04_Prediksi_dan_Evaluasi.xlsx':'Nilai guru, prediksi, RMSE, Pearson, QWK, dan Gwet AC2.',
    '05_Riwayat_Pelatihan.xlsx':'Seluruh fold dan epoch digabung per model.'}
for name in names:
    print('•',name)
    print('  ',descriptions.get(name,''))
if len(names)<5:
    print('Paket belum lengkap: hanya tahap yang sudah selesai yang disertakan.')
if '04_Prediksi_dan_Evaluasi.xlsx' in names:
    print('Mulai membaca hasil akhir dari 04_Prediksi_dan_Evaluasi.xlsx → Perbandingan Model.')
if IN_COLAB:
    from google.colab import files
    files.download(download['archive'])
else:print('Lokasi file:',download['archive'])
